# 09 — Image operations

**Workload:** Stride-trick convolution, downsampling, and image normalization.

This notebook is executed against the RNP engine. Every output below is
stored in the notebook and visible when rendered on GitHub.

In [1]:
from pathlib import Path
import importlib.util
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "shim" / "rnp_numpy").is_dir()
)
for path in (PROJECT_ROOT / "harness" / "_redirect", PROJECT_ROOT / "shim"):
    sys.path.insert(0, str(path))

# IPython may preload the oracle NumPy, so clear that namespace before
# executing the exact redirect hook used by examples/run_all.py.
for module_name in list(sys.modules):
    if module_name == "numpy" or module_name.startswith("numpy."):
        del sys.modules[module_name]
redirect_path = PROJECT_ROOT / "harness" / "_redirect" / "sitecustomize.py"
redirect_spec = importlib.util.spec_from_file_location("_rnp_notebook_redirect", redirect_path)
redirect = importlib.util.module_from_spec(redirect_spec)
redirect_spec.loader.exec_module(redirect)
import numpy as np

probe = np.array(0)
print("numpy version:", np.__version__)
print(f"RNP engine active: {np.__name__} ({type(probe).__module__}.{type(probe).__name__})")
assert np.__name__ == "rnp_numpy"

numpy version: 2.5.2
RNP engine active: rnp_numpy (_rnp.ndarray)


## Create a synthetic image

Combine a smooth ramp with a bright rectangular feature.

In [2]:
y, x = np.mgrid[0:8, 0:10]
image = 0.25 * x + 0.5 * y
image[2:6, 3:7] += 5.0
print("image shape:", image.shape)
print(image)

image shape: (8, 10)
[[0.   0.25 0.5  0.75 1.   1.25 1.5  1.75 2.   2.25]
 [0.5  0.75 1.   1.25 1.5  1.75 2.   2.25 2.5  2.75]
 [1.   1.25 1.5  6.75 7.   7.25 7.5  2.75 3.   3.25]
 [1.5  1.75 2.   7.25 7.5  7.75 8.   3.25 3.5  3.75]
 [2.   2.25 2.5  7.75 8.   8.25 8.5  3.75 4.   4.25]
 [2.5  2.75 3.   8.25 8.5  8.75 9.   4.25 4.5  4.75]
 [3.   3.25 3.5  3.75 4.   4.25 4.5  4.75 5.   5.25]
 [3.5  3.75 4.   4.25 4.5  4.75 5.   5.25 5.5  5.75]]


## Convolve with stride-trick windows

Apply a normalized 3×3 Gaussian-like kernel without a specialized image library.

In [3]:
kernel = np.array([[1.0, 2.0, 1.0], [2.0, 4.0, 2.0], [1.0, 2.0, 1.0]]) / 16.0
windows = np.lib.stride_tricks.sliding_window_view(image, (3, 3))
convolved = np.einsum("ijxy,xy->ij", windows, kernel)
print("convolution shape:", convolved.shape)

convolution shape: (6, 8)


## Downsample and normalize

Take every other filtered pixel and map the result to the unit interval.

In [4]:
downsampled = convolved[::2, ::2]
normalized = (downsampled - downsampled.min()) / np.ptp(downsampled)
print("downsampled image:\n", downsampled)
print("normalized image:\n", np.round(normalized, 4))
print("normalized range:", normalized.min(), normalized.max())

downsampled image:
 [[0.75   2.1875 3.     2.5625]
 [1.75   6.     7.75   4.5   ]
 [2.75   6.0625 7.5    5.1875]]
normalized image:
 [[0.     0.2054 0.3214 0.2589]
 [0.1429 0.75   1.     0.5357]
 [0.2857 0.7589 0.9643 0.6339]]
normalized range: 0.0 1.0


## Verify the result

In [5]:
expected = [
    [0.75, 2.1875, 3.0, 2.5625],
    [1.75, 6.0, 7.75, 4.5],
    [2.75, 6.0625, 7.5, 5.1875],
]
assert np.allclose(downsampled, expected, rtol=0.0, atol=0.0)
assert np.array_equal(np.array([normalized.min(), normalized.max()]), [0.0, 1.0])
print("PASS — all image-operation assertions passed.")

PASS — all image-operation assertions passed.
